## Import & Setup

In [1]:
import cv2
import os
import time
import pandas as pd
import numpy as np
from pyzbar import pyzbar

## 1. Threshold setting


In [2]:
# =========================
# THRESHOLD SETTINGS
# =========================
# BLUR_THRESHOLD = 60.0
# CONTRAST_THRESHOLD = 35.0
BRIGHTNESS_THRESHOLD = 90.0
BLUR_THRESHOLD = 40.0
CONTRAST_THRESHOLD = 25.0
# BRIGHTNESS_THRESHOLD = 80.0
ANGLE_PASS_MAX = 15
ANGLE_REJECT_MAX = 30

## 2. Read image


In [3]:
# def load_image(path, size=(640, 480)):
#     img = cv2.imread(path)
#     if img is None:
#         return None
#     img = cv2.resize(img, size)
#     return img

def load_image(path, max_size=1280):
    img = cv2.imread(path)
    if img is None:
        return None
    h, w = img.shape[:2]
    # Only downscale if too large, preserve small images
    if max(h, w) > max_size:
        scale = max_size / max(h, w)
        img = cv2.resize(img, (int(w * scale), int(h * scale)), 
                        interpolation=cv2.INTER_AREA)
    return img

## 3. Retinex + Preprocessing

In [4]:
# def simple_retinex(img, sigma=15):
#     img = img.astype(np.float32) + 1.0
#     blur = cv2.GaussianBlur(img, (0, 0), sigma)
#     retinex = np.log(img) - np.log(blur)
#     retinex = cv2.normalize(retinex, None, 0, 255, cv2.NORM_MINMAX)
#     return np.uint8(retinex)

# def preprocess_image(img):
#     gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
#     brightness = float(np.mean(gray))

#     # auto enhancement
#     if brightness < BRIGHTNESS_THRESHOLD:
#         enhanced = simple_retinex(gray, sigma=15)
#         method = "RETINEX"
#     else:
#         clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
#         enhanced = clahe.apply(gray)
#         method = "CLAHE"

#     _, otsu = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
#     adaptive = cv2.adaptiveThreshold(
#         enhanced,
#         255,
#         cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
#         cv2.THRESH_BINARY,
#         11,
#         2
#     )

#     return {
#         "gray": gray,
#         "enhanced": enhanced,
#         "otsu": otsu,
#         "adaptive": adaptive,
#         "brightness": brightness,
#         "method": method
#     }

In [5]:
def simple_retinex(img, sigma=15):
    img = img.astype(np.float32) + 1.0
    blur = cv2.GaussianBlur(img, (0, 0), sigma)
    retinex = np.log(img) - np.log(blur)
    retinex = cv2.normalize(retinex, None, 0, 255, cv2.NORM_MINMAX)
    return np.uint8(retinex)


def preprocess_image(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    brightness = float(np.mean(gray))

    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    clahe_img = clahe.apply(gray)

    retinex_img = simple_retinex(gray, sigma=15)

    if brightness < BRIGHTNESS_THRESHOLD:
        enhanced = retinex_img
        method = "RETINEX"
    else:
        enhanced = clahe_img
        method = "CLAHE"

    _, otsu = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    adaptive = cv2.adaptiveThreshold(
        enhanced,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        31,
        5
    )

    return {
        "gray": gray,
        "enhanced": enhanced,
        "clahe": clahe_img,
        "retinex": retinex_img,
        "otsu": otsu,
        "adaptive": adaptive,
        "brightness": brightness,
        "method": method
    }

## 4. Quality Inspection

In [6]:
def quality_inspection(preprocessed):
    enhanced = preprocessed["enhanced"]

    # blur
    blur = cv2.Laplacian(enhanced, cv2.CV_64F).var()

    # contrast / dynamic range
    contrast = float(enhanced.max() - enhanced.min())

    return {
        "blur": float(blur),
        "contrast": contrast
    }

## 5. Decision Logic 1

In [7]:
def decision_logic_1(quality_metrics):
    blur = quality_metrics["blur"]
    contrast = quality_metrics["contrast"]

    if blur < BLUR_THRESHOLD:
        return "REJECT", "BLUR"

    if contrast < CONTRAST_THRESHOLD:
        return "REJECT", "LOW_CONTRAST"

    return "PASS", "QUALITY_OK"

## 6. Qr detection & decode

In [8]:
def estimate_qr_angle(decoded_obj):
    pts = decoded_obj.polygon

    if pts is None or len(pts) < 2:
        return None

    p1 = pts[0]
    p2 = pts[1]

    angle = np.degrees(np.arctan2(p2.y - p1.y, p2.x - p1.x))

    if angle > 90:
        angle -= 180
    elif angle < -90:
        angle += 180

    return float(angle)



# def qr_detection(preprocessed):
#     enhanced = preprocessed["enhanced"]

#     decoded = pyzbar.decode(enhanced)

#     # =========================
#     # ANGLE CONFIG (ADD HERE)
#     # =========================
#     ANGLE_PASS_MAX = 15
#     ANGLE_REJECT_MAX = 30

#     if decoded:
#         qr_data = decoded[0].data.decode("utf-8", errors="ignore")
#         angle = estimate_qr_angle(decoded[0])

#         # =========================
#         # ANGLE DECISION (ADD HERE)
#         # =========================
#         if angle is not None:
#             a = abs(angle)

#             if a <= ANGLE_PASS_MAX:
#                 verdict = "PASS"
#             elif a >= ANGLE_REJECT_MAX:
#                 verdict = "REJECT"
#                 reason = "rotation"
#             else:
#                 verdict = "BORDERLINE"
#         else:
#             verdict = "UNKNOWN"

#         return {
#             "qr_found": True,
#             "qr_data": qr_data,
#             "angle": angle,
#             "verdict": verdict,
#             "decode_method": "PYZBAR"
#         }

#     return {
#         "qr_found": False,
#         "qr_data": None,
#         "angle": None,
#         "verdict": "REJECT",
#         "reason": "no_qr",
#         "decode_method": None
#     }
    

In [9]:
# def try_decode(img):
#     decoded = pyzbar.decode(img)
#     if decoded:
#         obj = decoded[0]
#         return {
#             "qr_found": True,
#             "qr_data": obj.data.decode("utf-8", errors="ignore"),
#             "angle": estimate_qr_angle(obj),
#             "decode_method": "PYZBAR"
#         }

#     detector = cv2.QRCodeDetector()
#     try:
#         text, points, _ = detector.detectAndDecode(img)
#         if text:
#             return {
#                 "qr_found": True,
#                 "qr_data": text,
#                 "angle": None,
#                 "decode_method": "OPENCV_QR"
#             }
#     except:
#         pass

#     return None


# def qr_detection(preprocessed):
#     candidates = []

#     gray = preprocessed["gray"]
#     enhanced = preprocessed["enhanced"]
#     otsu = preprocessed["otsu"]
#     adaptive = preprocessed["adaptive"]

#     # basic candidates
#     candidates.append(("GRAY", gray))
#     candidates.append(("ENHANCED", enhanced))
#     candidates.append(("OTSU", otsu))
#     candidates.append(("ADAPTIVE", adaptive))

#     # upscale candidates
#     for name, img in [("GRAY", gray), ("ENHANCED", enhanced), ("OTSU", otsu), ("ADAPTIVE", adaptive)]:
#         up = cv2.resize(img, None, fx=2.0, fy=2.0, interpolation=cv2.INTER_CUBIC)
#         candidates.append((name + "_UP2", up))

#     # small angle rotations
#     for angle in [-45,-30,-20, -10, 10, 20,30,45]:
#         h, w = enhanced.shape[:2]
#         M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
#         rotated = cv2.warpAffine(enhanced, M, (w, h), borderMode=cv2.BORDER_REPLICATE)
#         candidates.append((f"ROT_{angle}", rotated))

#     for method_name, img in candidates:
#         result = try_decode(img)
#         if result is not None:
#             result["decode_method"] = method_name + "_" + result["decode_method"]
#             return result

#     return {
#         "qr_found": False,
#         "qr_data": None,
#         "angle": None,
#         "verdict": "REJECT",
#         "reason": "no_qr",
#         "decode_method": "FAILED_ALL"
#     }

In [10]:
def qr_detection(preprocessed):
    candidates = []

    gray      = preprocessed["gray"]
    enhanced  = preprocessed["enhanced"]
    otsu      = preprocessed["otsu"]
    clahe_img = preprocessed["clahe"]
    retinex_img = preprocessed["retinex"]

    # === BASIC CANDIDATES ===
    candidates.append(("GRAY",    gray))
    candidates.append(("ENHANCED", enhanced))
    candidates.append(("OTSU",    otsu))
    candidates.append(("CLAHE",   clahe_img))
    candidates.append(("RETINEX", retinex_img))

    # === INVERTED (white-on-dark QRs) ===
    candidates.append(("INV_GRAY",     cv2.bitwise_not(gray)))
    candidates.append(("INV_ENHANCED", cv2.bitwise_not(enhanced)))
    candidates.append(("INV_OTSU",     cv2.bitwise_not(otsu)))

    # === SHARPENED ===
    def sharpen(img):
        blurred = cv2.GaussianBlur(img, (0, 0), 3)
        return cv2.addWeighted(img, 1.5, blurred, -0.5, 0)
    candidates.append(("SHARPENED", sharpen(enhanced)))

    # === DENOISED ===
    denoised = cv2.fastNlMeansDenoising(gray, h=10)
    candidates.append(("DENOISED", denoised))
    _, denoised_otsu = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    candidates.append(("DENOISED_OTSU", denoised_otsu))

    # === MULTIPLE ADAPTIVE BLOCK SIZES ===
    for block_size in [11, 21, 31, 51]:
        adap = cv2.adaptiveThreshold(enhanced, 255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, block_size, 5)
        candidates.append((f"ADAPTIVE_{block_size}", adap))

    # === UPSCALED CANDIDATES ===
    for name, img in [("GRAY", gray), ("ENHANCED", enhanced),
                    ("OTSU", otsu), ("CLAHE", clahe_img)]:
        up2 = cv2.resize(img, None, fx=2.0, fy=2.0, interpolation=cv2.INTER_CUBIC)
        candidates.append((f"{name}_UP2", up2))
        up3 = cv2.resize(img, None, fx=3.0, fy=3.0, interpolation=cv2.INTER_CUBIC)
        candidates.append((f"{name}_UP3", up3))

    # === ROTATION CANDIDATES ===
    for angle in [-45, -30, -20, -15, -10, -5, 5, 10, 15, 20, 30, 45]:
        h, w = enhanced.shape[:2]
        M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
        rotated = cv2.warpAffine(enhanced, M, (w, h), borderMode=cv2.BORDER_REPLICATE)
        candidates.append((f"ROT_{angle}", rotated))
        # upscale rotated too
        up = cv2.resize(rotated, None, fx=2.0, fy=2.0, interpolation=cv2.INTER_CUBIC)
        candidates.append((f"ROT_{angle}_UP2", up))

    # === TRY ALL CANDIDATES ===
    for method_name, img in candidates:
        result = try_decode(img)
        if result is not None:
            result["decode_method"] = method_name + "_" + result["decode_method"]
            return result

    return {
        "qr_found": False,
        "qr_data":  None,
        "angle":    None,
        "verdict":  "REJECT",
        "reason":   "no_qr",
        "decode_method": "FAILED_ALL"
    }

In [11]:
def try_decode(img):
    # 1. pyzbar first
    decoded = pyzbar.decode(img)
    if decoded:
        obj = decoded[0]
        return {
            "qr_found": True,
            "qr_data": obj.data.decode("utf-8", errors="ignore"),
            "angle": estimate_qr_angle(obj),
            "decode_method": "PYZBAR"
        }

    # 2. OpenCV standard
    detector = cv2.QRCodeDetector()
    try:
        text, points, _ = detector.detectAndDecode(img)
        if text:
            return {"qr_found": True, "qr_data": text, "angle": None, "decode_method": "OPENCV_QR"}
    except:
        pass

    # 3. WeChat (handles tilted, small, damaged QRs)
    try:
        wechat = cv2.wechat_qrcode_WeChatQRCode()
        res, _ = wechat.detectAndDecode(img)
        if res and res[0]:
            return {"qr_found": True, "qr_data": res[0], "angle": None, "decode_method": "WECHAT"}
    except:
        pass

    return None




In [12]:
# def is_valid_qr_data(data):
#     if data is None or len(data.strip()) == 0:
#         return False
#     if len(data) < 3:  # Too short to be meaningful
#         return False
#     # Check printable characters ratio
#     printable = sum(1 for c in data if c.isprintable())
#     return (printable / len(data)) > 0.8

# # In try_decode(), after getting qr_data:
# if not is_valid_qr_data(qr_data):
#     return None  # Treat as no QR found

# def try_decode(img):
#     decoded = pyzbar.decode(img)
#     if decoded:
#         obj = decoded[0]
#         qr_data = obj.data.decode("utf-8", errors="ignore")
#         if is_valid_qr_data(qr_data):  # ✅ ADD THIS CHECK
#             return {"qr_found": True, "qr_data": qr_data,
#                     "angle": estimate_qr_angle(obj), "decode_method": "PYZBAR"}

#     detector = cv2.QRCodeDetector()
#     try:
#         text, points, _ = detector.detectAndDecode(img)
#         if text and is_valid_qr_data(text):  # ✅ ADD THIS CHECK
#             return {"qr_found": True, "qr_data": text, "angle": None, "decode_method": "OPENCV_QR"}
#     except: pass

#     try:
#         wechat = cv2.wechat_qrcode_WeChatQRCode()
#         res, _ = wechat.detectAndDecode(img)
#         if res and res[0] and is_valid_qr_data(res[0]):  # ✅ ADD THIS CHECK
#             return {"qr_found": True, "qr_data": res[0], "angle": None, "decode_method": "WECHAT"}
#     except: pass

#     return None

def is_valid_qr_data(data):
    if data is None:
        return False

    data = data.strip()

    if len(data) < 3:
        return False

    # Reject weird characters
    printable = sum(1 for c in data if c.isprintable())
    ratio = printable / len(data)

    if ratio < 0.85:
        return False

    return True

## 7. Decision Logic 2

In [13]:
def decision_logic_2(qr_result):
    if qr_result["qr_found"]:
        return "PASS", "QR_DETECTED"

    return "NO_QR", "QR_NOT_DETECTED"

# def decision_logic_2(qr_result):
#     if not qr_result["qr_found"]:
#         return "NO_QR", "QR_NOT_DETECTED"

#     angle = qr_result.get("angle", None)

#     if angle is None:
#         return "REJECT", "NO_ANGLE"

#     if abs(angle) <= ANGLE_PASS_MAX:
#         return "PASS", "ANGLE_OK"

#     elif abs(angle) <= ANGLE_REJECT_MAX:
#         return "REJECT", "ANGLE_BORDERLINE"

#     else:
#         return "REJECT", "ANGLE_TOO_LARGE"

## 8. Run Dataset

In [14]:
def run_dataset(folder):
    results = []

    for file in os.listdir(folder):
        path = os.path.join(folder, file)

        if not os.path.isfile(path):
            continue

        t0 = time.perf_counter()

        img = load_image(path)
        if img is None:
            continue

        # Step 1: preprocessing
        prep = preprocess_image(img)

        # Step 2: quality inspection
        quality_metrics = quality_inspection(prep)

        # Step 3: decision logic 1
        quality_status, quality_reason = decision_logic_1(quality_metrics)

        # Step 4: QR detection
        qr_result = qr_detection(prep)

        # Step 5: decision logic 2
        qr_status, qr_reason = decision_logic_2(qr_result)

        latency_ms = (time.perf_counter() - t0) * 1000.0

        results.append({
            "file": file,
            "brightness": round(prep["brightness"], 2),
            "method": prep["method"],
            "blur": round(quality_metrics["blur"], 2),
            "contrast": round(quality_metrics["contrast"], 2),

            # result 1
            "quality_status": quality_status,
            "quality_reason": quality_reason,

            # qr result
            "angle": None if qr_result["angle"] is None else round(qr_result["angle"], 2),
            "qr_found": qr_result["qr_found"],
            "qr_data": qr_result["qr_data"],
            "decode_method": qr_result["decode_method"],

            # result 2
            "qr_status": qr_status,
            "qr_reason": qr_reason,

            "latency_ms": round(latency_ms, 2)
        })

    return pd.DataFrame(results)

## 9. Single Image Test

In [ ]:
# =========================
# TEST SINGLE IMAGE
# =========================

test_path = r"D:\UNIKL\FYP2\QR_data\train\images\spotsimage007.jpg"

img = load_image(test_path)

if img is None:
    print("Image not found!")
else:
    prep = preprocess_image(img)

    # quality inspection
    quality_metrics = quality_inspection(prep)
    quality_status, quality_reason = decision_logic_1(quality_metrics)

    # qr detection
    qr_result = qr_detection(prep)
    qr_status, qr_reason = decision_logic_2(qr_result)

    print("=== QUALITY RESULT ===")
    print("Blur:", round(quality_metrics["blur"], 2))
    print("Contrast:", round(quality_metrics["contrast"], 2))
    print("Status:", quality_status)
    print("Reason:", quality_reason)

    print("\n=== QR DETECTION RESULT ===")
    print("QR Found:", qr_result["qr_found"])
    print("QR Data:", qr_result["qr_data"])
    print("Angle:", qr_result["angle"])
    print("Status:", qr_status)
    print("Reason:", qr_reason)



In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,5))

plt.subplot(1,3,1)
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title("Original")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(prep["enhanced"], cmap='gray')
plt.title(f"Enhanced ({prep['method']})")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(prep["otsu"], cmap='gray')
plt.title("Otsu Threshold")
plt.axis("off")

plt.show()

## 10. Run and display

In [ ]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

folder_path = r"D:\UNIKL\FYP2\QR_data\train\images"

df = run_dataset(folder_path)
display(df)

## 11. Summary of Result 1


In [ ]:
print("QUALITY INSPECTION RESULT")
print(df["quality_status"].value_counts())

print("\nQUALITY REASON")
print(df["quality_reason"].value_counts())

## 12. Summary of Result 2

In [ ]:
print("QR DETECTION RESULT")
print(df["qr_status"].value_counts())

print("\nQR DETECTION REASON")
print(df["qr_reason"].value_counts())

## 13. Final Result

In [ ]:
# =========================
# FINAL RESULT (COMBINED)
# =========================
df["final_result"] = df.apply(
    lambda row: "PASS" if row["quality_status"] == "PASS" and row["qr_status"] == "PASS"
    else "REJECT",
    axis=1
)


print("\n=== FINAL RESULT SUMMARY ===")
print(df["final_result"].value_counts())


# =========================
# FINAL REASON
# =========================
df["final_reason"] = df.apply(
    lambda row: row["quality_reason"] if row["quality_status"] != "PASS"
    else row["qr_reason"],
    axis=1
)

print("\n=== FINAL REASON SUMMARY ===")
print(df["final_reason"].value_counts())


# =========================
# FINAL PERFORMANCE (%)
# =========================
total = len(df)
pass_count = (df["final_result"] == "PASS").sum()
reject_count = (df["final_result"] == "REJECT").sum()

print("\n=== FINAL PERFORMANCE ===")
print(f"Total Images : {total}")
print(f"PASS         : {pass_count} ({pass_count/total*100:.2f}%)")
print(f"REJECT       : {reject_count} ({reject_count/total*100:.2f}%)")

## 13. Save CSV

In [ ]:
df.to_csv("results_AI_wo_angle.csv", index=False)
print("Saved to results_AI_wo_angle.csv")

## 14. Evalution Metric

In [22]:
import os

output_path = r"D:\UNIKL\FYP2\QR_data\train\labels"

def annotation_label(filename):
    name = os.path.splitext(filename)[0]
    label_file = os.path.join(output_path, name + ".txt")

    if not os.path.exists(label_file):
        return "NO_QR"

    with open(label_file, "r") as f:
        lines = f.readlines()

    # if len(lines) > 0:
        # return "PASS"
    if len(lines) > 0 and any(line.strip() for line in lines):
        return "PASS"
    else:
        return "NO_QR"

In [23]:
df["actual_label"] = df["file"].apply(annotation_label)

In [ ]:
# =========================
# EVALUATION METRICS
# =========================

# Safety check
if "df" not in globals():
    print(" Error: df not found. Please run dataset cell first.")

else:
    print("=== CHECK LABELS ===")
    # print("Ground Truth:", df["ground_truth"].unique())
    print("Actual Label:", df["actual_label"].unique())
    print("Prediction  :", df["qr_status"].unique())

    # =========================
    # PREPARE DATA
    # =========================

    y_true = df["actual_label"]
    y_pred = df["final_result"]

    # Convert to binary (PASS = 1, others = 0)
    y_true_bin = y_true.apply(lambda x: 1 if x == "PASS" else 0)
    y_pred_bin = y_pred.apply(lambda x: 1 if x == "PASS" else 0)

    # =========================
    # CONFUSION MATRIX
    # =========================
    TP = ((y_true_bin == 1) & (y_pred_bin == 1)).sum()
    TN = ((y_true_bin == 0) & (y_pred_bin == 0)).sum()
    FP = ((y_true_bin == 0) & (y_pred_bin == 1)).sum()
    FN = ((y_true_bin == 1) & (y_pred_bin == 0)).sum()

    # =========================
    # METRICS
    # =========================
    total = TP + TN + FP + FN

    accuracy = (TP + TN) / total if total > 0 else 0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1_score = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0

    # =========================
    # OUTPUT
    # =========================
    print("\n=== CONFUSION MATRIX ===")
    print(f"TP (Correct PASS) : {TP}")
    print(f"TN (Correct NO_QR): {TN}")
    print(f"FP (False PASS)   : {FP}")
    print(f"FN (Missed QR)    : {FN}")

    print("\n=== METRICS ===")
    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1 Score  : {f1_score:.4f}")